# Importação e instalação

In [0]:
!pip install -q \
    "pydantic<2" spacy==3.7.4 \
    sentence-transformers==3.0.1 \
    numpy==1.26.4 \
    striprtf==0.0.22 \
    tqdm==4.66.5 \
    html2text \
    ftfy

In [0]:
!python -m spacy download pt_core_news_lg || python -m spacy download pt_core_news_sm

In [0]:
dbutils.library.restartPython()

In [0]:
# ==== Stdlib ====
import re, unicodedata, html as _html, json
from dataclasses import dataclass, asdict
from functools import lru_cache
from typing import List, Dict, Tuple, Any
from difflib import SequenceMatcher

# ==== Numérico / Dados ====
import numpy as np
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.5f' % x)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2000)


# ==== NLP / Embeddings ====
import spacy
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ==== RTF -> Texto (OK no PyPI) ====
from striprtf.striprtf import rtf_to_text

# ==== HTML -> Texto ====
from lxml.html import fromstring
import html2text

# ==== Unicode/Mojibake ====
import ftfy

# ==== Progresso ====
from tqdm import tqdm

In [0]:
# === Configuração do Spacy ===
nlp = spacy.load("pt_core_news_lg", disable=["ner"])
nlp.add_pipe("sentencizer", config={"punct_chars": ["\n"]}, before="parser")
nlp.max_length = 1500000

##Variáveis de Ambiente

In [0]:
dbutils.widgets.text("id_projeto", "colon", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
print("environment:", environment)

In [0]:
environment_tbl = "" if environment in ["hml", "prd"] else f"{environment}_"
print("environment_tbl:", environment_tbl)

In [0]:
import datetime

dbutils.widgets.text("data_execucao_modelo", "", "Data Execução Modelo")
data_execucao_modelo = dbutils.widgets.get("data_execucao_modelo")
if data_execucao_modelo == "":
    data_execucao_modelo = datetime.datetime.now().strftime("%Y-%m-%d")

In [0]:
INPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_colon"
OUTPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_colon_saida_conferencia_2"
CATALOG = "ia"

print('Environment:    ', environment_tbl)
print('Tabela entrada: ', INPUT_TABLENAME)
print('Tabela destino: ', OUTPUT_TABLENAME)
print('catalogo:       ', CATALOG)
print('Data Referencia:', data_execucao_modelo) # Modelo de data YYYY-MM-DD

# Dados de entrada

In [0]:
df_laudos_original_completo = spark.sql(f"""
                                select *
                                from {CATALOG}.{INPUT_TABLENAME}
                                where cast(dataExecucaoModelo as date) = date('{data_execucao_modelo}')
                                """).toPandas()
df_laudos_original_completo.shape

# Manipulacao e limpeza

In [0]:
def norm(s: str) -> str:
    s = s.lower().strip()
    s = ''.join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r"\s+", " ", s)
    return s

In [0]:
_HTML_BRK_TAGS = re.compile(r'</?(p|div|br|li|tr|h[1-6])\b[^>]*>', re.IGNORECASE)
_HTML_TAGS     = re.compile(r'<[^>]+>')
_HTML_SCRIPT   = re.compile(r'<(script|style)\b[^>]*>.*?</\1\s*>', re.IGNORECASE|re.DOTALL)
_HTML_DATAIMG  = re.compile(r'data:image/[^;]+;base64,[A-Za-z0-9+/=\s]+', re.IGNORECASE)
_HTML_JUNK_ATTR= re.compile(r'\sdata-[\w\-]+="[^"]*"')
_HTML_JUNK_CLS = re.compile(r'\sclass="[^"]*"')
_HTML_WIDGET   = re.compile(r'\s(?:aria-[\w\-]+|role|tabindex|contenteditable|spellcheck|draggable|style|width|height)="[^"]*"', re.IGNORECASE)
_UPPER_SPACED_RX = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)
_FONT_WORDS = {
    "times", "times new roman", "arial", "calibri", "cambria", "courier", "courier new",
    "helvetica", "symbol", "wingdings", "opensymbol", "ms mincho", "simsun", "century",
    "cambria math"
}

NEG_HINTS = (" sem ", " ausencia ", " nao ")
GENERIC_SINGLETON_BAN = {
    "livre","leito","difusa","difuso","vias","via","direito","esquerdo","normal",
    "discreto","discreta","helicoidal","multislice","aortoiliaca","contornos",
    "calibre","material","gasoso","fecal","porta","veia","arteria","hepatico",
    "hepatica","hilo","anastomose","fluxo","indice","resistencia","propria",
    "ramo","velocidade","pre","pos","maxima","mucosa","vascular","padrao",
    "padrao vascular","conteudo","homogeneo","integridade", "parietal"
}
CONFIG = {
    "negation": {
        "tokens": [
            "nao", "sem", "ausencia de", "negado", "nega", "descarta", "livre de", "sem evidencias de",
            "sem sinais de", "sem achados de", "sem alteraçoes", "sem alteracoes", "sem sinais", "sem achados",
            "sem alteracao", "sem alteraçao", "sem polipos", "sem diverticulos"
        ],
        "window_tokens": 6  # alcance simples de negação
    },
    "organs": {
        # Cada órgão: seeds e (opcional) regex adicionais
        "figado": {
            "seeds": ["figado", "hepatico", "hepatica", "lobos hepaticos", "parenquima hepatico", "hepatopatia"],
            "regex": [r"\bhepat\w*"]
        },
        "rim": {
            "seeds": ["rim", "renal", "parênquima renal", "pielocalicial", "cortex renal"],
            "regex": [r"\bren(al|ais)\b", r"\brim(es)?\b"]
        },
        "vesicula biliar": {
            "seeds": ["vesicula", "vesicula biliar", "colecisto", "colecistica"],
            "regex": [r"\bcolec\w*"]
        },
        "colon_reto": {
            "seeds": [
                # Termos gerais
                "colon", "cólon",
                # Segmentos do cólon (encontrados nos laudos)
                "colon ascendente", "cólon ascendente", "colon transverso", "cólon transverso",
                "colon descendente", "cólon descendente", "colon sigmoide", "cólon sigmoide",
                # Ceco e apêndice (MUITO FREQUENTES - 72x apêndice, 70x apêndice cecal)
                "ceco", "apendice", "apêndice", "apendice cecal", "apêndice cecal",
                "apendice retrocecal", "apêndice retrocecal", "apendice vermiforme", "apêndice vermiforme",
                # Válvula ileocecal
                "valvula ileocecal", "válvula ileocecal", "valvula ileo cecal", "válvula íleo cecal",
                "valvula ileo-cecal", "válvula íleo-cecal",
                # Sigmoide
                "sigmoide", "sigmóide", "sigmoide redundante", "sigmóide redundante",
                # Reto
                "reto", "reto distal", "reto medio", "reto médio", "reto proximal",
                # Transição
                "transicao retossigmoide", "transição retossigmoide", "transicao reto-sigmoide", "transição reto-sigmoide",
                "juncao retossigmoide", "junção retossigmoide",
                # Alças e estruturas (encontrados nos laudos)
                "alças colônicas", "alças colônicas", "alcas colonicas", "alças do colon", "alças do cólon",
                "parede colonica", "parede colônica",
                # Divertículos (FREQUENTES - 20x, termo comum em tomografias)
                "diverticulos", "divertículos", "diverticulos colonicos", "divertículos colônicos",
                "diverticulos colicos", "divertículos cólicos", "diverticulose", "diverticulose colonica", "diverticulose colônica"
            ],
            "regex": [
                r"\bc[oó]l[oô]n\b",  # cólon ou colon (SEM "oscopia")
                r"\breto(s)?\b",  # reto ou retos
                r"\bsigm[oó]ide\b",  # sigmoide ou sigmóide
                r"\b(ascendente|transverso|descendente)\b",  # segmentos
                r"\bceco\b",  # ceco
                r"\bap[eê]ndice( cecal| retrocecal| vermiforme)?\b",  # apêndice e variações
                r"\bv[aá]lvula (i|í)leo ?-?cecal\b",  # válvula ileocecal
                r"\btransi[cç][aã]o reto-?sigm[oó]ide\b",  # transição retossigmoide
                r"\bjun[cç][aã]o retossigm[oó]ide\b",  # junção retossigmoide
                r"\bal[cç]as col[oô]nicas\b",  # alças colônicas
                r"\bparede col[oô]nica\b",  # parede colônica
                r"\bdivert[ií]culos?( col[oô]nicos?| c[oó]licos?)?\b",  # divertículos e variações
                r"\bdiverticulose( col[oô]nica)?\b",  # diverticulose
            ]
        }
    ,
        # Adicione novos órgãos aqui
    },
    "findings": {
        # Achados por órgão (canônicos + seeds). A expansão semântica cria sinônimos dinâmicos no runtime
        "figado": {
            "esteatose": ["esteatose", "infiltracao gordurosa", "gordura hepatica", "esteatose", "doenca hepatica gordurosa", "gordura hepatica", "ecogenicidade aumentada", "hiperecogenicidade difusa", "ecotextura aumentada", "ecogenicidade parenquimatosa aumentada", "ecotextura hepatica aumentada", "ecotextura parenquimatosa aumentada", "ecogenicidade difusamente aumentada", "hipoatenuacao difusa", "atenuacao reduzida difusa"],
            "infiltracao_gordurosa": ["infiltracao gordurosa"],
            "fibrose": ["fibrose", "fibrose hepatica", "fibrose avancada", "fibrose cirrosica", "fibrose cirrosica avancada"],
            "nodulo": ["nodulo", "nódulo", "nódulos", "nódulos hepáticos", "nodularidade", "nodular", "nodulares"],
            "cirrose": ["cirrose", "cirrose hepatica", "cirrosica", "cirrose avancada"],
            "hepatomegalia": ["hepatomegalia", "figado aumentado"],
            "trombose_veia_porta": ["trombose da veia porta", "trombose portal", "trombose da veia portal"],
            "baco_aumentado": ["baco aumentado"],
            "cavernoma_porta": ["cavernoma porta"],
            "circulacao_colateral": ["circulacao colateral"],
            "congestao_passiva_cronica_figado": ["congestao passiva cronica figado", "congestao passiva cronica"],
            "contorno_irregular": ["contorno irregular"],
            "degeneracao_gordurosa_figado": ["degeneracao gordurosa figado", "degeneracao gordurosa"],
            "dilatacao_vias_biliares": ["dilatacao vias-biliares"],
            "doenca_alcoolica_figado": ["doenca alcoolica figado", "doenca alcoolica"],
            "doenca_hepatica": ["doenca hepatica", "contorno lobulado", "contorno levemente lobulado"],
            "encefalopatia_hepatica": ["encefalopatia hepatica"],
            "esclerose_hepatica": ["esclerose hepatica"],
            "fibrose_esclerose_alcoolicas": ["fibrose esclerose alcoolicas"],
            "figado_gorduroso_alcoolico": ["figado gorduroso alcoolico"],
            "hepatite": ["hepatite"],
            "hipertensao_portal": ["hipertensao portal"],
            "hipertrofia_lobo_caudado": ["hipertrofia lobo caudado"],
            "infarto_figado": ["infarto figado"],
            "insuficiencia_hepatica_alcoolica": ["insuficiencia hepatica alcoolica"],
            "insuficiencia_hepatica": ["insuficiencia hepatica"],
            "lirads_4": ["lirads 4", "categoria 4", "categ 4", "rads 4"],
            "lirads_5": ["lirads 5", "categoria 5", "categ 5", "rads 5"],
            "lobulado": ["lobulado", "lobuloso", "lobulos"],
            "necrose_hemorragica": ["necrose hemorragica"],
            "nodulo_hipervascular": ["nodulo hipervascular", "nodular hipervascularizado"],
            "reduzido": ["reduzido", "diminuido"],
            "sindrome_obstrucao_sinusoidal_hepatica": ["sindrome obstrucao sinusoidal hepatica"],
            "varizes_esofagianas": ["varizes esofagianas"],
            "varizes_gastricas": ["varizes gastricas"],
            "veia_porta_dilatada": ["veia porta dilatada"],
            "hepatopatia_cronica": ["hepatopatia cronica"],
            "microlitiase": ["microlitiase"],
            "multiplos_calculos": ["multiplos calculos"]
        },
        "rim": {
            "hidronefrose": ["hidronefrose", "dilatacao pielocalicial", "sistema coletor dilatado"],
            "litiases": ["calculo renal", "litiases renais", "nefrolitiase"]
        },
        "vesicula biliar": {
            "colelitíase": ["calculo em vesicula", "colelitíase", "calculose vesicular"],
            "colecistite": ["colecistite", "parede espessada", "sinal de murphy ultrassonografico"]
        },
        "colon_reto": {
        "lesao": [
            "lesao", "lesoes", "lesao vegetante", "lesao infiltrativa", "lesao exofitica", "lesao deprimida",
            "estenose por lesao", "lesao suspeita", "lesao ulcerada", "areas de necrose", "tecido suspeito",
            "angiectasia", "angiodisplasia"
        ],
        "polipo": [
            "polipo", "polipos", "polipo pediculado", "polipo sessil","polipo sésil","polipo sesseil",
            "lesao plana", "adenoma", "lesao sessil","lesao séssil", "lesao pediculada",
            "paris 0-ii", "paris 0-iia", "paris 0-iib", "paris 0-iic", "paris 0-is",
            "polipo suspeito", "polipos suspeitos"
        ],
        "ulceracao":[
            "ulcera", "ulcero", "ulcerativa", "ulcero infitrativa", "ulcero-infiltrativa", "ulceracao", "ulceracoes", "erosao", "erosoes",
            "erosiva", "afoide", "aftoide", "aftoides"
        ],
        "tumor_ou_massa": [
            "massa", "tumor", "tumoracao", "tumoracoes", "tumoracao exofitica", "nodulo", "nodular", "nodulacao", "nodulacoes",
            "laterally spreading tumor", "lst", "lst-g", "lst-ng", "neoplasia", "adeoncarcinoma", "carcinoma", "neoplasica", 
            "neoplasico", "nodulares", "metastase"
        ],
        "espessamento":[
            "rigidez", "espessamento"
        ],
        "sangramento":[
            "friabilidade", "sangramento"
        ],
        "contorno_ou_margem":[
            "alteracao irregular", "margem irregulares", "margens irregulares", "parede irregular", "alteracao focal", "alteracao de contorno", "bordas irregulares",
            "crescimento heterogeneo", "opacidade heterogenea", "densidade heterogenea", "escavacoes", "perda de margens claras", "perda do padrao",
            "lateralmente extensa", "margem mal definida", "margem indistinta", "contorno irregular"
        ]
        #"neoplasia_confirmada": [
        #    "adenocarcinoma", "carcinoma", "carcinoma in situ", "neoplasia maligna", "neoplasia invasiva"
        #],
        #"displasia_alto_grau": [
        #    "displasia de alto grau", "alto grau de displasia", "hgd"
        #],
        #"hiperplasia_linfoide": [
        #    "hiperplasia linfoide", "hiperplasia linfoide reacional"
        #],
        #"diverticulose": [
        #    "diverticulos", "diverticulose", "ostios diverticulares"
        #],
        #"hemorroidas": [
        #    "hemorroida", "hemorroidas", "botoes hemorroidarios", "botões hemorroidários"
        #],
        },
    },
    # Limiar de similaridade para incluir novos termos do texto no léxico semântico
    "semantic": {
        "min_sim_seed_vocab": 0.85,  # quão parecido um termo do vocabulário do laudo precisa ser com os seeds
        "top_k_per_seed": 8,
        # Geramos candidatos do vocabulário via noun_chunks + n-grams leves
        "min_token_len": 3
    }
}

TARGET_ORGAN = "colon_reto"
FORCE_FULL_DOC_FOR = set()
SPACY_STOP = set(w.lower() for w in nlp.Defaults.stop_words)
OPERATORS = {"não", "nao", "sem", "há", "ha"}
OPERATORS |= set(CONFIG["negation"]["tokens"])

def _norm_plain(s: str) -> str:
    import unicodedata, re
    s = s.lower().strip()
    s = ''.join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r"\s+", " ", s)
    return s

STOP_WORDS_NORM = { _norm_plain(w) for w in SPACY_STOP }
OPERATORS_NORM  = { _norm_plain(w) for w in OPERATORS }

# remove operadores do conjunto de stopwords
STOP_WORDS_NORM = STOP_WORDS_NORM - OPERATORS_NORM
HEADER_RX = re.compile(r'^\s*[-•]?\s*([A-ZÁ-Úa-zá-ú/ ]{2,})\s*:\s*', re.UNICODE)

HEADER_ALIASES = {
    "figado": {"figado", "fígado", "fígado ", "figado "},
    "rim": {"rim", "rins"},
    "vesicula biliar": {"vesicula", "vesícula", "vesicula biliar", "vesícula biliar"},
    # adicione aliases para outros cabeçalhos que existam no laudo (ex.: vias biliares, pancreas, adrenais, etc.)
    "vias biliares": {"vias biliares", "vias biliares intra e extra-hepaticas", "vias biliares intra e extra-hepáticas"},
    "adrenais": {"adrenais", "supra-renais", "suprarrenais"},
    "pancreas": {"pancreas", "pâncreas"},
    "bases pulmonares": {"bases pulmonares", "pulmoes", "pulmões"},
    "conclusao": {"conclusao", "conclusão"},
    "colon_reto": {
        # Termos gerais
        "colon", "cólon",
        # Segmentos do cólon
        "colon ascendente", "cólon ascendente", "colon transverso", "cólon transverso",
        "colon descendente", "cólon descendente", "colon sigmoide", "cólon sigmoide",
        # Ceco e apêndice (apendice removido)
        "ceco",
        # Válvula ileocecal
        "valvula ileocecal", "válvula ileocecal", "valvula ileo cecal", "válvula íleo cecal",
        "valvula ileo-cecal", "válvula íleo-cecal",
        # Sigmoide
        "sigmoide", "sigmóide", "sigmoide redundante", "sigmóide redundante",
        # Reto
        "reto", "reto distal", "reto medio", "reto médio", "reto proximal",
        # Transição
        "transicao retossigmoide", "transição retossigmoide", "transicao reto-sigmoide",
        "transição reto-sigmoide", "juncao retossigmoide", "junção retossigmoide",
        # Alças e estruturas
        "alças colônicas", "alças colônicas", "alcas colonicas", "alças do colon",
        "alças do cólon", "parede colonica", "parede colônica",
        # Divertículos
        "diverticulos", "divertículos", "diverticulos colonicos", "divertículos colônicos",
        "diverticulos colicos", "divertículos cólicos", "diverticulose",
        "diverticulose colonica", "diverticulose colônica"
    }
}

NEG_SET = {norm(x) for x in CONFIG["negation"]["tokens"]}


In [0]:
@dataclass
class FindingHit:
    organ: str
    finding: str
    surface: str
    start_char: int
    end_char: int
    negated: bool
    confidence: float

# @lru_cache(maxsize=8192)

In [0]:
df_laudos_original_completo['data_limpa'] = df_laudos_original_completo['dataexame'].astype(str).str.replace('-', '') ##<<lake
df_laudos_original_completo['record_id'] = df_laudos_original_completo['data_limpa'] + df_laudos_original_completo['id_pct'] ##<< lake

df_laudos_original_completo = df_laudos_original_completo.dropna(subset=['data_limpa'])

if df_laudos_original_completo.shape[0] == 0:
    dbutils.notebook.exit("OK")
else:
    display(df_laudos_original_completo)

In [0]:
df_laudos = df_laudos_original_completo
df_laudos.shape

# Utilitarios

In [0]:
def unique(seq):
    seen=set(); out=[]
    for x in seq:
        if x not in seen:
            out.append(x); seen.add(x)
    return out

In [0]:
def _rtf_to_text_fallback(rtf: str, debug: bool = False) -> str:
    if not rtf:
        return ""

    import re, unicodedata

    # ---- helpers locais ----
    def _undo_py_escapes(s: str) -> str:
        # reverte escapes que o Python injeta quando a string não é raw
        return (s
            .replace("\x0c", r"\f")   # form feed -> \f
            .replace("\t",  r"\tab")  # tab -> \tab (token RTF real)
        )

    def _strip_rtf_metadata_top(s: str) -> str:
        # Remove SOMENTE blocos padrão logo no topo (primeiros ~8 KB)
        head = s[:8192]
        tail = s[8192:]

        # cada bloco: remove no máximo 1 ocorrência por padrão
        patterns = [
            r"{\\\*?\\fonttbl\b[^{}]*}",
            r"{\\\*?\\colortbl\b[^{}]*}",
            r"{\\\*?\\stylesheet\b[^{}]*}",
            r"{\\\*?\\listtable\b[^{}]*}",
            r"{\\\*?\\listoverridetable\b[^{}]*}",
            r"{\\\*?\\generator\b[^{}]*}",
            r"{\\\*?\\info\b[^{}]*}",
        ]
        for pat in patterns:
            head = re.sub(pat, " ", head, flags=re.IGNORECASE)

        return head + tail

    def _drop_rtf_meta_lines(text: str) -> str:
        out = []
        only_words_semicolon_rx = re.compile(r'^[A-Za-z0-9 /+\-\*\u00C0-\u017F]+;\s*$')
        font_tokens = r"(unicode|opensymbol|wingdings|monospaced|serif|sans|arial|calibri|times|courier)"
        # aceita " * " OU "\* " antes do nome da fonte
        font_line_rx = re.compile(
            rf'^\s*[A-Za-z0-9 \-\u00C0-\u017F]+(?:\s\\?\*\s*)?(?:{font_tokens})(?:\s*[;:]?)\s*$', re.IGNORECASE)
        tiny_meta_rx = re.compile(r'^\s*(default|\*jword2|\* generator|\* info)\s*[;:]?\s*$', re.IGNORECASE)

        for ln in text.splitlines():
            lns = ln.strip()
            if not lns:
                out.append(ln); continue
            if only_words_semicolon_rx.match(lns):
                continue
            if font_line_rx.match(lns):
                continue
            if tiny_meta_rx.match(lns):
                continue
            # fallback bem específico: linha curta com qualquer token de fonte + ';'
            if ';' in lns and re.search(font_tokens, lns, re.IGNORECASE) and len(lns) <= 80:
                continue
            out.append(ln)
        return "\n".join(out).strip()

    # -------------------------

    s = _undo_py_escapes(rtf)
    # normaliza EOL cedo
    s = s.replace("\r\n", "\n").replace("\r", "\n")

    # pega só se realmente parece RTF
    if not s.lstrip().startswith("{\\rtf"):
        # não é RTF: devolve como veio
        return s.strip()

    # tira metadados do topo de forma conservadora
    s = _strip_rtf_metadata_top(s)

    if debug:
        print("--- after strip_top ---")
        print(s[:1000])

    # 1) CP1252: \'hh
    def _hex_sub(m):
        try:
            return bytes([int(m.group(1), 16)]).decode('cp1252', errors='ignore')
        except Exception:
            return ""
    s = re.sub(r"\\'([0-9a-fA-F]{2})", _hex_sub, s)

    # 2) Unicode: \uNNNN?
    def _u_sub(m):
        try:
            code = int(m.group(1))
            if code < 0:
                code += 65536
            return chr(code)
        except Exception:
            return ""
    s = re.sub(r"\\u(-?\d+)\??", _u_sub, s)

    # 3) Quebras e tabs
    s = re.sub(r"\\par[d]?\b", "\n", s, flags=re.IGNORECASE)
    s = re.sub(r"\\line\b", "\n", s, flags=re.IGNORECASE)
    s = re.sub(r"\\tab\b", "\t", s, flags=re.IGNORECASE)

    # 4) Remove destinos conhecidos (restantes, pontuais)
    s = re.sub(r"{\\\*?\\(fonttbl|colortbl|stylesheet|info|generator|listtable|listoverridetable)[^}]*}",
               " ", s, flags=re.IGNORECASE)

    # 5) Remover comandos de controle (apenas os que começam com barra)
    #    Mais conservador: exige backslash + letras (com opcional número)
    #s = re.sub(r"\\[a-zA-Z]+-?\d*(?:\s|;)?", " ", s)
    s = re.sub(r"\\[a-zA-Z]+-?\d*\s?", " ", s)

    # 6) Escapes literais e chaves
    s = s.replace("\\{", "{").replace("\\}", "}").replace("\\\\", "\\")
    s = re.sub(r"[{}]", " ", s)  # neste ponto já decodificamos conteúdo útil

    # 7) Plano B: 'hh sem barra (alguns dumps)
    s = re.sub(r"(?<!\\)'([0-9a-fA-F]{2})", _hex_sub, s)

    # 8) Limpa tokens RTF residuais comuns (bem restritivo)
    s = re.sub(r"\b(?:rtf\d*|ansi|ansicpg\d+|deftab\d+|paper[wh]\d+|marg[tlrb]\d+|headery\d+|footery\d+|colsx?\d+|snext\d+|fs\d+|cf\d+|pard|plain|qc|ql|itap\d+|viewkind\d+|uc\d+|sa\d+|sl\d+|slmult\d+|lang\d+|kerning\d+|ulnone|b0|i0|f\d+|fs\d+|cf\d+|kerning\d+|deff\d+|colortbl|fonttbl|stylesheet|info|generator|listtable|listoverridetable)\b",
           " ", s, flags=re.IGNORECASE)

    # 9) Normalização
    s = unicodedata.normalize("NFKC", s)
    # remove invisíveis
    s = s.replace("\u200b", "").replace("\u200c", "").replace("\xa0", " ")
    # espaços
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\s*\n\s*", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s).strip()

    # 10) Derruba linhas-meta finais
    s = _drop_rtf_meta_lines(s)

    # Garantia mínima: se sobrar vazio, devolve original “bruto” decodificado por CP1252
    if not s:
        s = bytes(rtf, "latin1", errors="ignore").decode("cp1252", errors="ignore").strip()

    return s

In [0]:
def remove_final_laudo(texto: str) -> str:
    if not texto:
        return ""

    # --- REMOÇÕES GLOBAIS INLINE (apareçam onde aparecerem) ---

    # 1) "Laudo Gerado/Lib[e]rado por Sistema Especialista."
    texto = re.sub(r'(?i)\blaudo\s+(?:gerado|liberado)\s+por\s+um?\s*sistema\s+especialista\b\.?', ' ', texto)

    # 2) "Para visualização do conteúdo do laudo, favor/… acessar/acesse … Viewer/opção imagem/imagem disponível"
    texto = re.sub(r'(?i)\bpara\s+visualiza\w*\s+(?:o\s+conte[uú]do\s+do\s+)?laudo\b.*?\b(?:favor\s+)?(?:acesse|acessar)\b.*?\b(?:viewer|op[cç][aã]o\s+imagem|imagem\s+dispon[ií]vel)\b\.?', ' ', texto)

    # 3) "Laudo pode não estar completo na visualização em RTF/Imagem"
    texto = re.sub(r'(?i)\blaudo\s+pode\s+na[oã]\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+(?:rtf|imagem)\b\.?', ' ', texto)

    # 4) "Referências: - Endoscopic Classification Review Group..." (e variações)
    # Busca por "referencia(s)" e remove o restante do texto.
    texto = re.sub(r'(?i)\brefer[eê]ncia[s]?\s*[:\-\.]?.*$', ' ', texto, flags=re.DOTALL)

    # --- REMOÇÃO POR SENTENÇA (linhas inteiras de aviso/rodapé) ---
    pats_norm = [
        r"laudo\s+pode\s+nao\s+estar\s+completo.*?\b(rtf|imagem)\b",
        r"para\s+visualiza\w+.*?\b(rtf|imagem|sistema|webris|viewer)\b",
        r"acessar\s+a\s+op[cç][aã]o\s+imagem",
        r"sistema\s+especialista",
        r"sistema\s+da\s+radiologia\s+webris",
        # novas combinações explícitas:
        r"este\s+laudo\s+foi\s+liberado\s+por\s+um?\s+sistema\s+especialista",
        r"favor\s+acessar\s+a\s+op[cç][aã]o\s+imagem\s+dispon[ií]vel",
        r"visualiza\w+\s+do\s+conteudo\s+do\s+laudo",
        r"laudo\s+pode\s+nao\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+rtf",
    ]
    rx = re.compile(r"^(?:\s*(?:{})(?:[.!?])?\s*)$".format("|".join(pats_norm)), re.IGNORECASE)

    sents = re.split(r'(?<=[.!?])\s+|\n+', texto.strip())
    kept = []
    for s in sents:
        s_norm = norm(s)
        if not rx.match(s_norm):
            kept.append(s)

    out = " ".join(kept)
    out = re.sub(r'\s{2,}', ' ', out)
    out = re.sub(r'\s+([.!?,;:])', r'\1', out)
    return out.strip()

In [0]:
def to_plain(s: str) -> str:
    """
    Conversão robusta (RTF/HTML/texto) -> texto limpo:
      1) detecta RTF/HTML e usa parser dedicado (striprtf / lxml / html2text)
      2) ftfy para arrumar unicode/mojibake
      3) normalizações e correções de espaços intra-palavra/acento
      4) limpeza de boilerplate típico de RTF/HTML
      5) remoção de rodapés conhecidos e ajuste de pontuação
    """
    if not s:
        return ""

    raw = s.replace("\r\n", "\n").replace("\r", "\n")

    # --- 1) Detectar formato ---
    head = raw[:512].lstrip()
    is_rtf  = head.startswith("{\\rtf") or "\\rtf" in head.lower()[:128]
    is_html = bool(re.search(r"</?(html|head|body|div|p|span|br|table|tr|td|img)\b", raw[:2000], re.I))

    txt = raw
    try:
        if is_rtf:
            try:
                # Tenta usar a biblioteca externa (rtf_to_text)
                txt = rtf_to_text(raw)
            except Exception:
                # Fallback para a função interna se a biblioteca falhar
                txt = _rtf_to_text_fallback(raw)
        elif is_html:
            # ... (lógica HTML)
            pass
        else:
            txt = raw
    except Exception:
        # fallback bruto caso algo quebre
        txt = raw

    # --- 2) Unicode/encodes (corrige mojibake, NBSP etc.) ---
    try:
        txt = ftfy.fix_text(txt)
    except Exception:
        pass

    # invisíveis e normalização
    txt = (txt.replace("\u200b", "")
              .replace("\u200c", "")
              .replace("\xa0", " "))
    txt = unicodedata.normalize("NFKC", txt)

    # --- 3) Colapsar cabeçalhos ALL CAPS espaçados (CON C LU S Ã O) ---
    RX_CAPS_LINE   = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)
    RX_CAPS_INLINE = re.compile(r'(?<!\w)([A-ZÁ-Ú](?:\s+[A-ZÁ-Ú]){2,})(?!\w)', re.UNICODE)
    def _collapse_caps(line: str) -> str:
        if RX_CAPS_LINE.match(line.strip()):
            return line.replace(" ", "")
        return RX_CAPS_INLINE.sub(lambda m: m.group(1).replace(" ", ""), line)
    txt = "\n".join(_collapse_caps(ln) for ln in txt.splitlines())

    # --- 4) Remover prefixo numérico de OCR grudado (ex.: "6reto" -> "reto"), sem afetar dosagens ---
    UNITS = {
        "mg","g","kg","mcg","ug","μg","ml","l","dl","cl","mm","cm","m","km",
        "bpm","mmhg","pa","ua","u/l","ui","iu","u","meq","mol","mmol","nmol",
        "na","k","cl","co2","o2","sat","spo2","fio2","cr","vdrl","ph"
    }
    RX_DOSAGE = re.compile(r'(?i)^\d{1,3}\s*(?:' + r'|'.join(sorted(UNITS, key=len, reverse=True)) + r')\b')
    RX_FLOW   = re.compile(r'(?i)^\d{1,3}\s*[lL]\s*/\s*min\b')
    RX_OCR_PREFIX = re.compile(r'(?i)\b(\d{1,2})([A-Za-zÁ-Úá-ú]{3,})\b')
    def _strip_digit_prefix(m: re.Match) -> str:
        tok = m.group(0)
        if RX_DOSAGE.match(tok) or RX_FLOW.match(tok) or m.group(2).lower() in UNITS:
            return tok
        return m.group(2)
    txt = RX_OCR_PREFIX.sub(_strip_digit_prefix, txt)

    # --- 5) Corrigir espaços intra-palavra (acentos/ç quebrados) ---
    ACC = "ÁÉÍÓÚÂÊÎÔÛÃÕÄËÏÖÜáéíóúâêîôûãõäëïöüÇç"
    for _ in range(4):
        before = txt

        # letras acentuadas separadas do resto
        txt = re.sub(rf'(?i)\b([{ACC}])\s+([A-Za-z]{{1,30}})\b', r'\1\2', txt)
        txt = re.sub(rf'(?i)\b([A-Za-z]{{1,30}})\s+([{ACC}][A-Za-z]{{0,30}})\b', r'\1\2', txt)

        # trinca com acento no meio: "p ó s" -> "pós"
        txt = re.sub(rf'(?i)\b([A-Za-z])\s+([{ACC}])\s+([A-Za-z])\b', r'\1\2\3', txt)

        # sufixos comuns
        txt = re.sub(r'(?i)çã\s+o\b',  'ção',  txt)
        txt = re.sub(r'(?i)çõ\s+es\b', 'ções', txt)
        txt = re.sub(r'(?i)ã\s+o\b',   'ão',   txt)
        txt = re.sub(r'(?i)õ\s+es\b',  'ões',  txt)

        # hífen com espaços indevidos
        txt = re.sub(rf'(?i)\b([A-Za-z{ACC}]{{1,40}})\s*-\s*([A-Za-z{ACC}]{{1,40}})\b', r'\1-\2', txt)

        # casos de espaço antes de letras acentuadas dentro de palavra (cola sempre)
        txt = re.sub(rf'([A-Za-z])\s+(?=[{ACC}])', r'\1', txt)
        txt = re.sub(rf'([{ACC}])\s+(?=[A-Za-z])', r'\1', txt)

        if txt == before:
            break

    # --- 6) Limpeza de boilerplate herdado de RTF/HTML ---
    lines = []
    for ln in txt.splitlines():
        s = ln.strip()
        sl = s.lower()
        if not s:
            lines.append(ln); continue
        # lixos clássicos de RTF/HTML
        if any(tok in sl for tok in ("fonttbl","colortbl","stylesheet","generator",
                                     "heading","listoverridetable","listtable")):
            continue
        if s.count(';') >= 2 and re.search(r'(arial|times|calibri|courier|symbol|wingdings)', sl):
            continue
        # linhas só de pontuação/traços
        if re.fullmatch(r'[;,\.\-–—\s]+', s):
            continue
        lines.append(ln)
    txt = "\n".join(lines)

    # --- 7) Espaços/pontuação finais e quebras ---
    txt = re.sub(r'[ \t]+', ' ', txt)
    txt = re.sub(r'\s*\n\s*', '\n', txt)
    txt = re.sub(r'\n{3,}', '\n\n', txt)
    txt = re.sub(r'\s+([.,;:])', r'\1', txt)
    txt = re.sub(r'([\(\[\{])\s+', r'\1', txt)
    txt = re.sub(r'\s+([\)\]\}])', r'\1', txt)

    # --- 8) Remover rodapés/avisos de visualização incompleta e limpar pontas ---
    txt = remove_final_laudo(txt).strip()
    return txt

In [0]:
def extract_keys_unique(dict_list):
    seen = set(); out = []
    for d in dict_list:
        if isinstance(d, dict) and d:
            k = next(iter(d))
            if k not in seen:
                seen.add(k); out.append(k)
    return out

In [0]:
def _force_sentence_breaks(text: str) -> str:
    """
    Força quebras de sentença em pontos finais seguidos de maiúscula.
    """
    import re
    # Adiciona quebra de linha após ponto + espaço + maiúscula
    text = re.sub(r'\.(\s+)([A-Z])', r'.\n\2', text)
    return text

In [0]:
def _resolve_target_organs(target, organs_cfg):
    if target is None:
        return list(organs_cfg.keys())
    if isinstance(target, str):
        target = [target]
    out = []
    for o in target:
        k = o.strip().lower()
        if k not in organs_cfg:
            raise ValueError(f"Órgão desconhecido: {o}. Disponíveis: {list(organs_cfg.keys())}")
        out.append(k)
    return out

In [0]:
def build_doc_candidates(doc, min_len=3) -> list[str]:
    cand = []

    # 1) noun_chunks: mantemos inteiros (úteis como termos compostos reais do laudo)
    for nc in doc.noun_chunks:
        t = norm(nc.text)
        if len(t) >= min_len and not t.isnumeric():
            cand.append(t)

    # 2) n-grams: antes de montar, filtramos stopwords (com negadores preservados)
    toks = [norm(t.text) for t in doc if t.is_alpha and len(norm(t.text)) >= min_len]
    # STOP_WORDS_NORM deve ter sido criado antes, removendo OPERATORS (negadores)
    toks = [t for t in toks if t not in STOP_WORDS_NORM]

    # 3) gerar n-grams (1–3) a partir dos tokens filtrados
    for n in (1, 2, 3):
        for i in range(len(toks) - n + 1):
            cand.append(" ".join(toks[i:i+n]))

    # 4) dedup e retorno
    return unique([c for c in cand if len(c) >= min_len])

In [0]:
def fuzzy_sim(a: str, b: str) -> float:
    # similaridade 0..1 (robusto a typos simples)
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

In [0]:
def token_set(s: str) -> set[str]:
    return set([t for t in norm(s).split() if t and t not in STOP_WORDS_NORM])

In [0]:
def semantic_expand(seeds: List[str], vocab: List[str],
                    min_sim: float, top_k: int) -> Dict[str, List[Tuple[str, float]]]:
    if not seeds or not vocab:
        return {s: [] for s in seeds}

    seed_norm  = [norm(s) for s in seeds]
    seed_emb   = embedder.encode(seed_norm, convert_to_numpy=True, normalize_embeddings=True)
    vocab_norm = [norm(v) for v in vocab]
    vocab_emb  = embedder.encode(vocab_norm, convert_to_numpy=True, normalize_embeddings=True)

    out = {s: [] for s in seeds}
    for i, s in enumerate(seeds):
        s_tokens = token_set(s)  # tokens normalizados da seed
        pairs = []

        for j, v in enumerate(vocab):
            emb_sim = float(np.dot(seed_emb[i], vocab_emb[j]))
            fz_sim  = fuzzy_sim(s, v)
            score   = 0.75 * emb_sim + 0.25 * fz_sim
            if score < min_sim:
                continue

            v_tokens = token_set(v)
            overlap = len(v_tokens & s_tokens)

            # Regra FLEXÍVEL de interseção:
            # - Se a similaridade vetorial é ALTA (>= 0.70), aceita mesmo sem interseção explícita
            # - Se é BAIXA/MÉDIA, exige interseção (>=1 token). Para seeds multi-termo, exige >=2.
            if emb_sim < 0.70:
                if len(s_tokens) >= 2:
                    if overlap < 2:
                        continue
                else:
                    if overlap < 1:
                        continue

            pairs.append((v, score))

        pairs.sort(key=lambda x: x[1], reverse=True)
        out[s] = pairs[:top_k]
    return out

In [0]:
def merge_semantic_sets(seed_list: List[str], sem_expanded: Dict[str, List[Tuple[str,float]]]) -> List[str]:
    terms = list(seed_list)
    for s, pairs in sem_expanded.items():
        for term, sim in pairs:
            if term not in terms:
                terms.append(term)
    return unique(terms)

In [0]:
def _line_spans_text(text: str):
    spans, off = [], 0
    for line in text.splitlines(True):
        spans.append((off, off+len(line), line))
        off += len(line)
    return spans

In [0]:
def _norm_header_name(s: str) -> str:
    return norm(s.replace(":", "").replace("-", " ").strip())

In [0]:
def segment_by_headers(doc, target_organs: list[str]) -> list[dict]:
    """
    Delimita blocos pelo NOME: do cabeçalho.
    Fecha o bloco no PRÓXIMO cabeçalho (qualquer nome), mesmo que haja texto na mesma linha do cabeçalho.
    """
    text = doc.text
    lines = _line_spans_text(text)

    # 1) capture TODOS os cabeçalhos (nome e posição de início)
    all_headers = []  # [(start_char, header_name_norm)]
    for (s, e, line) in lines:
        m = HEADER_RX.match(line)
        if m:
            name = _norm_header_name(m.group(1))
            all_headers.append((s, name))

    if not all_headers:
        return []

    # 2) quais cabeçalhos são do(s) órgão(s) alvo?
    wanted = set()
    for o in target_organs:
        wanted |= HEADER_ALIASES.get(o, {o})

    organ_headers = [(s, name) for (s, name) in all_headers if any(name.startswith(_norm_header_name(w)) for w in wanted)]

    if not organ_headers:
        return []

    # 3) blocos: [header alvo i, próximo header global)
    all_headers_sorted = sorted(all_headers, key=lambda x: x[0])
    blocks = []
    for s, name in sorted(organ_headers, key=lambda x: x[0]):
        # próximo cabeçalho global após 's'
        next_starts = [S for (S, _) in all_headers_sorted if S > s]
        end = next_starts[0] if next_starts else len(text)
        span = doc.char_span(s, end, alignment_mode="expand")
        if span is not None and (span.end - span.start) > 0:
            # mapeia de volta para o órgão alvo (pega o primeiro que bate)
            mapped = None
            for o in target_organs:
                if any(name.startswith(_norm_header_name(w)) for w in HEADER_ALIASES.get(o, {o})):
                    mapped = o
                    break
            blocks.append({"organ": mapped or target_organs[0], "span": span})
    return blocks

In [0]:
def organ_anchors(doc: "spacy.tokens.Doc", organ_name: str, organ_cfg: Dict[str, Any],
                  organ_lexicon: List[str]) -> List[Tuple[int,int]]:
    # Retorna spans (start_token, end_token) que ancoram menções ao órgão
    import re
    spans = []
    organ_terms = set([norm(organ_name)] + [norm(x) for x in organ_cfg.get("seeds", [])] + [norm(x) for x in organ_lexicon])
    
    for sent in doc.sents:
        s = norm(sent.text)
        hit = False
        
        # Verificar seeds com word boundaries
        for t in organ_terms:
            # Escapar caracteres especiais de regex
            t_escaped = re.escape(t)
            # Usar word boundary (\b) para match exato de palavras
            pattern = r'\b' + t_escaped + r'\b'
            if re.search(pattern, s):
                hit = True
                break
        
        # Se não encontrou via seeds, tentar regex
        if not hit and organ_cfg.get("regex"):
            for rx in organ_cfg["regex"]:
                if re.search(rx, s):
                    hit = True
                    break
        
        if hit:
            spans.append((sent.start, sent.end))
    
    return spans

In [0]:
def segment_by_organs(doc: "spacy.tokens.Doc",
                      organs_cfg: Dict[str, Any],
                      organ_lexicons: Dict[str, List[str]]) -> List[Dict[str, Any]]:
    blocks = []
    for organ, cfg in organs_cfg.items():
        anchors = organ_anchors(doc, organ, cfg, organ_lexicons.get(organ, []))
        for (a,b) in anchors:
            # bloco: da sentença-âncora até antes da próxima âncora de qualquer órgão (simples: apenas a sentença-âncora)
            span = doc[a:b]
            blocks.append({"organ": organ, "span": span})
    return blocks

In [0]:
def term_is_valid_for_seeds(term: str, seeds: list[str]) -> bool:
    toks = token_set(term)
    if not toks:
        return False

    # se houver seed multi-termo, exija candidato >=2 tokens
    if any(len(token_set(s)) >= 2 for s in seeds) and len(toks) < 2:
        return False

    # singletons banidos
    if len(toks) == 1 and next(iter(toks)) in GENERIC_SINGLETON_BAN:
        return False

    # sobreposição "literal"
    seed_tokens = set().union(*[token_set(s) for s in seeds])
    overlap_plain = len(toks & seed_tokens)

    if overlap_plain >= 1:
        return True

    # ---- raiz morfológica leve (inline, sem função nova) ----
    def _rootize_token(t: str) -> str:
        x = norm(t)
        # plurais/variações simples
        x = re.sub(r'(oes|ões|s)$', '', x, flags=re.IGNORECASE)
        x = re.sub(r'(ario|ária|ario(s)?|ária(s)?|ário|ária|ários|árias)$', '', x, flags=re.IGNORECASE)
        return x

    term_root = {_rootize_token(t) for t in toks}
    seed_root = set().union(*[{_rootize_token(t) for t in token_set(s)} for s in seeds])

    overlap_root = len(term_root & seed_root)
    return overlap_root >= 1

In [0]:
def _starts_with_negator(term: str) -> bool:
    toks = norm(term).split()
    return len(toks) > 0 and toks[0] in NEG_SET

In [0]:
def best_fuzzy_in_vocab(term: str, vocab: list[str], min_ratio: float = 0.86) -> str | None:
    # retorna o termo do vocabulário mais parecido acima do limiar
    best, best_s = None, 0.0
    for v in vocab:
        s = fuzzy_sim(term, v)
        if s > best_s:
            best, best_s = v, s
    return best if best_s >= min_ratio else None

In [0]:
def _accent_rx(t: str) -> str:
    rep = {
        'a': '[aáàâãä]', 'e': '[eéèêë]', 'i': '[iíìîï]',
        'o': '[oóòôõö]', 'u': '[uúùûü]', 'c': '[cç]', 'n': '[nñ]'
    }
    out = []
    for ch in t.lower():
        out.append(rep.get(ch, re.escape(ch)))
    return ''.join(out)

In [0]:
def _find_organ_mentions_in_text(text: str, organ_name: str, organ_cfg: Dict[str, Any], organ_lexicon: List[str]) -> List[Tuple[int, int]]:
    """
    Encontra todas as posições onde o órgão é mencionado no texto.
    Retorna lista de tuplas (start, end) com as posições das menções.
    """
    import re
    mentions = []
    text_norm = norm(text)
    
    # Seeds
    organ_terms = set([norm(organ_name)] + [norm(x) for x in organ_cfg.get("seeds", [])] + [norm(x) for x in organ_lexicon])
    
    for term in organ_terms:
        term_escaped = re.escape(term)
        pattern = r'\b' + term_escaped + r'\b'
        for m in re.finditer(pattern, text_norm):
            mentions.append((m.start(), m.end()))
    
    # Regex
    if organ_cfg.get("regex"):
        for rx_pattern in organ_cfg.get("regex"):
            for m in re.finditer(rx_pattern, text_norm):
                mentions.append((m.start(), m.end()))
    
    return mentions

In [0]:
def _finding_is_near_organ_mention(finding_pos: int, organ_mentions: List[Tuple[int, int]], max_distance: int = 200) -> bool:
    """
    Verifica se o finding está próximo (dentro de max_distance caracteres) de alguma menção ao órgão.
    """
    if not organ_mentions:
        return False
    
    for mention_start, mention_end in organ_mentions:
        dist_to_start = abs(finding_pos - mention_start)
        dist_to_end = abs(finding_pos - mention_end)
        min_dist = min(dist_to_start, dist_to_end)
        
        if min_dist <= max_distance:
            return True
    
    return False

In [0]:
def _extract_snippet_around_finding(doc_text: str, start: int, end: int, window: int = 100) -> str:
    """
    Extrai trecho ao redor do finding
    """
    snippet_start = max(0, start - window)
    snippet_end = min(len(doc_text), end + window)
    return doc_text[snippet_start:snippet_end]

In [0]:
def _snippet_mentions_other_organs(snippet: str, target_organ: str, all_organs_config: Dict) -> bool:
    """
    Verifica se o snippet menciona OUTROS órgãos (diferentes do target_organ)
    Retorna True se menciona APENAS outros órgãos (não menciona o target)
    """
    import re
    
    menciona_target = False
    menciona_outros = False
    
    # Verificar cada órgão configurado
    for organ_name, organ_cfg in all_organs_config.items():
        # Seeds do órgão
        organ_terms = set([norm(organ_name)] + [norm(x) for x in organ_cfg.get("seeds", [])])
        
        # Verificar se o snippet menciona este órgão
        menciona_este_orgao = False
        for term in organ_terms:
            # Word boundary para evitar falsos positivos
            pattern = r'\b' + re.escape(term) + r'\b'
            if re.search(pattern, norm(snippet)):
                menciona_este_orgao = True
                break
        
        # Classificar
        if menciona_este_orgao:
            if organ_name == target_organ:
                menciona_target = True
            else:
                menciona_outros = True
    
    # Retorna True se menciona APENAS outros (não menciona target)
    return menciona_outros and not menciona_target

In [0]:
def is_negated_in_sentence(doc,
                           start_char: int, end_char: int,
                           neg_list: list[str],
                           window_tokens: int = 12) -> bool:
    """
    True se houver negador NA MESMA SENTENÇA:
      - à ESQUERDA do termo (ex.: "não se observa ...", "nega ...")
      - à DIREITA do termo (ex.: "... sem alterações", "... ausência de ...")
    Usa a MESMA lista `neg_list` (derivada do CONFIG), sem dicionários extras.
    """
    hit = doc.char_span(start_char, end_char, alignment_mode="expand")
    if hit is None:
        return False
    sent = hit.sent

    # normaliza negadores uma vez
    neg_norm = [norm(x) for x in neg_list]

    # 0) se o próprio hit já começar por negador (raro, mas válido)
    hit_norm = norm(hit.text)
    if any(hit_norm.startswith(n + " ") or hit_norm == n for n in neg_norm):
        return True

    # 1) janela à esquerda
    left_tokens  = [t.text for t in sent if t.idx <  hit.start_char]
    left_tokens  = left_tokens[-window_tokens:] if window_tokens > 0 else left_tokens
    left_norm    = norm(" ".join(left_tokens))
    if any(n in left_norm for n in neg_norm):
        return True

    # 2) janela à direita
    right_tokens = [t.text for t in sent if t.idx >= hit.end_char]
    right_tokens = right_tokens[:window_tokens] if window_tokens > 0 else right_tokens
    right_norm   = norm(" ".join(right_tokens))
    if any(n in right_norm for n in neg_norm):
        return True

    return False

In [0]:
def match_findings_in_block(block: Dict[str,Any],
                            finding_cfg: Dict[str, List[str]],
                            neg_tokens: List[str], neg_window: int,
                            min_sim_seed_vocab: float,
                            organ_cfg: Dict[str, Any] = None,
                            organ_lexicon: List[str] = None) -> List[FindingHit]:
    span = block["span"]
    text_norm = norm(span.text)
    hits: List[FindingHit] = []
    organ_name = block["organ"]

    # vocabulário local do bloco para expansão
    local_vocab = build_doc_candidates(span, min_len=CONFIG["semantic"]["min_token_len"])

    for f_can, seeds in finding_cfg.items():
        # 0) seeds originais SEMPRE entram (normalizados e deduplicados)
        base_seeds = unique([norm(s) for s in seeds if s and norm(s)])

        # 1) expansão semântica (gera TERMOS, não inclui seeds)
        sem = semantic_expand(seeds, local_vocab, min_sim_seed_vocab, CONFIG["semantic"]["top_k_per_seed"])

        # antes: você simplesmente adicionava todos os termos de 'sem'
        # agora: usa os MESMOS filtros já aplicados em fuzzy
        sem_terms = []
        for _s, pairs in sem.items():
            for term, _sc in pairs:
                t = norm(term)
                if (t 
                    and t not in sem_terms
                    and t not in base_seeds
                    and term_is_valid_for_seeds(t, base_seeds)   # exige interseção literal/raiz com as seeds
                    and not _starts_with_negator(t)):
                    sem_terms.append(t)

        # 2) typos reais do texto via fuzzy (somente gerados)
        fuzzy_terms = set()
        for seed in seeds:
            hit = best_fuzzy_in_vocab(seed, local_vocab, min_ratio=0.86)
            if hit:
                fuzzy_terms.add(norm(hit))

        # 3) filtrar APENAS os fuzzy por qualidade/negador; sem_terms entram como vieram
        filtered_fuzzy = [
            t for t in fuzzy_terms
            if t not in base_seeds
            and term_is_valid_for_seeds(t, base_seeds)
            and not _starts_with_negator(t)
        ]

        # 4) léxico final = seeds + semânticos + fuzzy filtrados
        lexicon = unique(base_seeds + sem_terms + list(filtered_fuzzy))

        # 5) localizar termos no TEXTO (normalizado) e aplicar negação por sentença
        neg_list_norm = [norm(x) for x in neg_tokens]
        for term in lexicon:
            pat = r"(?i)(?<!\w)" + _accent_rx(term) + r"(?!\w)"
            rx = re.compile(pat)

            for m in rx.finditer(text_norm):
                abs_start = int(span.start_char + m.start())
                abs_end   = int(span.start_char + m.end())

                # VALIDAÇÃO DE CONTEXTO: verificar se o finding está próximo de menção ao órgão
                if organ_cfg is not None and organ_lexicon is not None:
                    organ_mentions = _find_organ_mentions_in_text(span.text, organ_name, organ_cfg, organ_lexicon)
                    finding_pos_in_block = abs_start - span.start_char
                    is_near = _finding_is_near_organ_mention(finding_pos_in_block, organ_mentions, max_distance=200)
                    
                    if not is_near:
                        continue
                    
                    # NOVA VALIDAÇÃO GENERALISTA: Verificar se menciona APENAS outros órgãos
                    snippet = _extract_snippet_around_finding(span.doc.text, abs_start, abs_end, window=100)
                    if _snippet_mentions_other_organs(snippet, organ_name, CONFIG["organs"]):
                        # Menciona apenas outros órgãos, não o target → REJEITAR
                        continue

                # negação: mesma sentença, à esquerda, janela limitada
                neg = is_negated_in_sentence(
                    span.doc,
                    start_char=abs_start,
                    end_char=abs_end,
                    neg_list=neg_list_norm,
                    window_tokens=neg_window
                )

                hits.append(FindingHit(
                    organ=block["organ"],
                    finding=f_can,
                    surface=text_norm[m.start():m.end()],
                    start_char=abs_start,
                    end_char=abs_end,
                    negated=neg,
                    confidence=0.9 if not neg else 0.35
                ))
    return hits

In [0]:
def extract_section_lines(text: str, header_name: str) -> list[str]:
    """Linhas (brutas) da seção `header_name` até o próximo cabeçalho."""
    lines = _line_spans_text(text)
    headers = []
    for (s,e,line) in lines:
        m = HEADER_RX.match(line)
        if m:
            headers.append((s, _norm_header_name(m.group(1))))
    if not headers:
        return []
    target = _norm_header_name(header_name)
    starts = [s for (s,name) in headers if name.startswith(target)]
    if not starts:
        return []
    s0 = starts[0]
    nexts = [S for (S,_) in headers if S > s0]
    s1 = nexts[0] if nexts else len(text)
    sub = text[s0:s1]
    return [ln.strip() for _,_,ln in _line_spans_text(sub)]

In [0]:
def _pretty_finding_name(f: str) -> str:
    return f.replace("_", " ").strip()

In [0]:
def _any_q_in_line(line_norm: str, queries_norm: set[str]) -> bool:
    # casa por token, não só por substring exata
    if not queries_norm:
        return False
    for q in queries_norm:
        if not q: 
            continue
        # aceita q inteiro ou seus tokens (ex.: "nodulo hipervascular" -> "nodulo" OU "hipervascular")
        toks = [t for t in q.split() if len(t) >= 4] or [q]
        if any(t in line_norm for t in toks):
            return True
    return False

In [0]:
def _shorten_clause(line_raw: str, query_norm: str, max_chars: int = 160) -> str:
    """
    Devolve o menor trecho útil dentro da linha que contém a query:
    - tenta cortar por pontuação (. ; : – ) ( )
    - se ainda ficar longo, usa uma janela em torno do match
    """
    ln = line_raw.strip()
    if len(ln) <= max_chars:
        return ln

    # 1) tenta quebrar por cláusulas
    parts = re.split(r'([.;:–\-\(\)])', ln)  # mantém separadores
    # remonta pares (texto+sep) para preservar sentenças curtas
    chunks = []
    cur = ""
    for p in parts:
        cur += p
        if p in ".;:–-)" and cur.strip():
            chunks.append(cur.strip())
            cur = ""
    if cur.strip():
        chunks.append(cur.strip())

    # procura a cláusula que contém a query (normalizada)
    q = norm(query_norm)
    for c in chunks:
        if q and norm(c).find(q) != -1:
            return c if len(c) <= max_chars else c[:max_chars].rsplit(" ", 1)[0] + "…"

    # 2) janela em torno do match (acentos-insensível)
    pat = _accent_rx(q) if q else None
    if pat:
        m = re.search(pat, ln, flags=re.IGNORECASE)
        if m:
            left = max(0, m.start() - max_chars//2)
            right = min(len(ln), m.end() + max_chars//2)
            snippet = ln[left:right].strip()
            # aparar nas bordas para próximo espaço
            if left > 0:
                snippet = "…" + snippet.split(" ", 1)[-1]
            if right < len(ln):
                snippet = snippet.rsplit(" ", 1)[0] + "…"
            return snippet

    # fallback
    return (ln[:max_chars].rsplit(" ", 1)[0] + "…") if len(ln) > max_chars else ln

In [0]:
def compact_summary_from_hits(hits: list[dict], doc, organ_blocks: list[dict]) -> list[dict]:
    # 1) só positivos
    pos = [h for h in hits if not h.get("negated", False)]
    if not pos:
        return []

    # 2) Conclusão — RAW e NORM
    concl_lines_raw = extract_section_lines(doc.text, "conclusao")
    concl_lines_norm = [norm(ln) for ln in concl_lines_raw]

    # 3) Fallback por bloco do órgão — RAW e NORM
    organ_texts = {b["organ"]: doc.text[b["span"].start_char:b["span"].end_char] for b in organ_blocks}
    organ_lines_raw = {org: [ln.strip() for _,_,ln in _line_spans_text(txt)]
                       for org, txt in organ_texts.items()}
    organ_lines_norm = {org: [norm(ln) for ln in lines]
                        for org, lines in organ_lines_raw.items()}

    # 4) agrupa por achado
    by_f = {}
    for h in pos:
        by_f.setdefault(h["finding"], []).append(h)

    out = []
    for f, hs in by_f.items():
        # ordena por confiança desc e span menor
        hs.sort(key=lambda h: (-h.get("confidence", 0.0), h["end_char"] - h["start_char"]))

        # queries: surface + tokens do nome do achado + padrões auxiliares
        queries = set()
        for h in hs:
            s = norm(h.get("surface",""))
            if s:
                queries.add(s)
                # também unigrams da surface (pega "nodulo", "hipervascularizado")
                for t in s.split():
                    if len(t) >= 4:
                        queries.add(t)
        # nome canônico
        fname = _pretty_finding_name(f)               # ex.: "nodulo hipervascular"
        queries.add(norm(fname))
        for t in norm(fname).split():
            if len(t) >= 4:
                queries.add(t)

        # heurística: se é achado nodular, garanta termos gerais
        if "nodulo" in queries or "nodule" in queries:
            queries |= {"nodulo", "nodular", "lirads", "li-rads"}

        best_line = None
        best_q = ""

        # 4.1 Conclusão (preferência forte) — devolve RAW encurtado
        for i, ln_norm in enumerate(concl_lines_norm):
            if _any_q_in_line(ln_norm, queries) and not any(h in ln_norm for h in NEG_HINTS):
                best_line = concl_lines_raw[i]
                # escolhe a query mais discriminativa para cortar
                # prioriza "lirads" ou "nodulo"
                if "lirads" in ln_norm:
                    best_q = "lirads"
                elif "nodulo" in ln_norm:
                    best_q = "nodulo"
                else:
                    # pega a maior query que exista na linha (mais específica)
                    best_q = max([q for q in queries if q in ln_norm], key=len, default="")
                best_line = _shorten_clause(best_line, best_q, max_chars=160)
                break

        # 4.2 Fallback: bloco do órgão — RAW encurtado
        if best_line is None:
            organ = hs[0]["organ"]
            onorm = organ_lines_norm.get(organ, [])
            oraw  = organ_lines_raw.get(organ, [])
            for i, ln_norm in enumerate(onorm):
                if _any_q_in_line(ln_norm, queries):
                    # query mais específica presente na linha
                    present = [q for q in queries if q in ln_norm]
                    qbest = max(present, key=len) if present else ""
                    best_line = _shorten_clause(oraw[i], qbest, max_chars=160)
                    break

        # 4.3 Último recurso: extrair contexto do documento usando a posição do hit
        if best_line is None:
            # Pegar o hit com maior confiança
            best_hit = hs[0]
            start_char = best_hit["start_char"]
            end_char = best_hit["end_char"]
            
            # Encontrar a sentença que contém o hit
            finding_sent = None
            for sent in doc.sents:
                if sent.start_char <= start_char < sent.end_char:
                    finding_sent = sent
                    break
            
            if finding_sent:
                # Usar a sentença completa
                sent_text = finding_sent.text.strip()
                
                # Se a sentença for muito longa, encurtar em torno do hit
                if len(sent_text) > 200:
                    # Calcular posição relativa do hit na sentença
                    hit_pos_in_sent = start_char - finding_sent.start_char
                    
                    # Janela de 100 chars antes e depois
                    window_start = max(0, hit_pos_in_sent - 100)
                    window_end = min(len(sent_text), hit_pos_in_sent + 100)
                    
                    snippet = sent_text[window_start:window_end].strip()
                    
                    # Adicionar reticências se cortou
                    if window_start > 0:
                        snippet = "…" + snippet.split(" ", 1)[-1] if " " in snippet else snippet
                    if window_end < len(sent_text):
                        snippet = snippet.rsplit(" ", 1)[0] + "…" if " " in snippet else snippet + "…"
                    
                    best_line = snippet
                else:
                    best_line = sent_text
            else:
                # Fallback: extrair janela em torno do hit
                window_start = max(0, start_char - 100)
                window_end = min(len(doc.text), end_char + 100)
                snippet = doc.text[window_start:window_end].strip()
                
                # Adicionar reticências
                if window_start > 0:
                    snippet = "…" + snippet
                if window_end < len(doc.text):
                    snippet = snippet + "…"
                
                best_line = snippet

        out.append({ _pretty_finding_name(f): best_line })
    return out

In [0]:
def process_report(text: str, target_organ=None) -> Dict[str, Any]:
    """
    target_organ: None | 'figado' | ['figado','rim',...]
    Executa a análise apenas para o(s) órgão(s) selecionado(s).
    """
    text = to_plain(text)  # já inclui cleanup + remove_final_laudo
    text = _force_sentence_breaks(text)
    doc = nlp(text)

    # 1) resolve órgãos alvo
    target_organs = _resolve_target_organs(
        target_organ if target_organ is not None else TARGET_ORGAN,
        CONFIG["organs"]
    )

    # 2) vocabulário global do documento (expansão semântica dos órgãos alvo)
    doc_vocab = build_doc_candidates(doc, min_len=CONFIG["semantic"]["min_token_len"])

    organ_lexicons = {}
    for organ in target_organs:
        ocfg = CONFIG["organs"][organ]
        sem = semantic_expand(
            ocfg.get("seeds", []),
            doc_vocab,
            CONFIG["semantic"]["min_sim_seed_vocab"],
            CONFIG["semantic"]["top_k_per_seed"]
        )
        organ_lexicons[organ] = merge_semantic_sets(ocfg.get("seeds", []), sem)

    # 3) segmentação por cabeçalho (preferencial). Se não achar, cai na sentença-âncora.
    #organs_subset_cfg = {k: CONFIG["organs"][k] for k in target_organs}

    #header_blocks = segment_by_headers(doc, target_organs)
    #if header_blocks:
    #    blocks = header_blocks
    #else:
        # fallback: sentença-âncora (sua função já existente)
    #    blocks = segment_by_organs(doc, organs_subset_cfg, organ_lexicons)

    # 3) segmentação
    organs_subset_cfg = {k: CONFIG["organs"][k] for k in target_organs}

    # NORMALIZAÇÃO do FORCE_FULL_DOC_FOR (aceita string ou set/list)
    _force_set = set()
    if isinstance(FORCE_FULL_DOC_FOR, str) and FORCE_FULL_DOC_FOR.strip():
        _force_set = {norm(FORCE_FULL_DOC_FOR)}
    else:
        try:
            _force_set = {norm(x) for x in (FORCE_FULL_DOC_FOR or set())}
        except TypeError:
            _force_set = set()

    # Força somente se o órgão-alvo estiver no FORCE_FULL_DOC_FOR
    force_full = any(norm(o) in _force_set for o in target_organs)

    if force_full:
        main_organ = next((o for o in target_organs if norm(o) in _force_set), target_organs[0])
        blocks = [{"organ": main_organ, "span": doc[:]}]
    else:
        header_blocks = segment_by_headers(doc, target_organs)
        if header_blocks:
            blocks = header_blocks
        else:
            blocks = segment_by_organs(doc, organs_subset_cfg, organ_lexicons)

    # 4) matching de achados apenas para os órgãos alvo
    all_hits: List[FindingHit] = []
    for b in blocks:
        organ = b["organ"]
        finding_cfg = CONFIG["findings"].get(organ, {})
        if not finding_cfg:
            continue
        hits = match_findings_in_block(
            b, finding_cfg,
            neg_tokens=[norm(x) for x in CONFIG["negation"]["tokens"]],
            neg_window=CONFIG["negation"]["window_tokens"],
            min_sim_seed_vocab=CONFIG["semantic"]["min_sim_seed_vocab"],
            organ_cfg=CONFIG["organs"].get(organ, {}),
            organ_lexicon=organ_lexicons.get(organ, [])
        )
        all_hits.extend(hits)

    # CORREÇÃO: Mover para FORA do loop for b in blocks
    findings_dicts = [asdict(h) for h in all_hits]
    result = {
        "organs_requested": target_organs,
        "organs_detected": unique([blk["organ"] for blk in blocks]),
        "findings": findings_dicts,
        "relevant": any(not h.negated for h in all_hits),
    }

    # resumo compacto
    sc = compact_summary_from_hits(findings_dicts, doc, blocks)

    # --- filtro: remove chaves com valores nulos/vazios e descarta dicts que ficarem vazios ---
    def _is_useful(v):
        if v is None:
            return False
        if isinstance(v, str):
            x = v.strip().lower()
            return x not in {"", "null", "none"}
        return True

    sc_clean = []
    for d in sc:
        clean = {k: v for k, v in d.items() if _is_useful(v)}
        if clean:
            sc_clean.append(clean)

    result["summary_compact"] = sc_clean
    return result

#Executa Modelo

In [0]:
#tratamento de laudos
df_laudos["laudo_tratado"] = df_laudos["Laudo"].map(to_plain)

In [0]:
#execução do modelo
laudos = df_laudos["laudo_tratado"].tolist()
resultados = df_laudos["laudo_tratado"].map(lambda s: (process_report(s, target_organ='colon_reto') or {}).get("summary_compact", []))

In [0]:
#extração de chaves do resultado
df_laudos["resultado"] = resultados
df_laudos["achados"] = df_laudos["resultado"].apply(lambda x: extract_keys_unique(x) if isinstance(x, list) else "")

In [0]:
#flag para laudos relevantes
df_laudos['fl_relevante'] = df_laudos['achados'].apply(lambda x: 1 if x != [] else 0)

#Tabela saida

In [0]:
colunas = [
    'record_id',
    'id_pct',
    'idunidade',
    'unidade',
    'paciente',
    'cpf',
    'sexo',
    'nascimento',
    'idade_anos',
    'telefonePacienteDDD',
    'telefonePaciente',
    'dataexame',
    'modalidade',
    'exame',
    'tipoexame',
    'an',
    'idstatus',
    'DataLaudoLiberado',
    'Laudo',
    'laudo_tratado',
    'resultado',
    'achados',
    'fl_relevante',
    'dataExecucaoModelo'
]

# reorganizar dataframe
df_laudos = df_laudos.reindex(columns=colunas)

In [0]:
df_laudos = df_laudos.rename(columns={'paciente':'pct_nome',
                        'cpf':'pct_cpf',
                        'sexo':'pct_sexo',
                        'nascimento':'pct_datanasc',
                        'idade_anos':'idade_paciente',
                        'dataexame':'exm_data',
                        'modalidade':'exm_mod',
                        'exame':'exm_titulo',
                        'tipoexame':'exm_tipo',
                        'an':'exm_an',
                        'idstatus':'exm_status',
                        'DataLaudoLiberado':'exm_laudo_dataliber',
                        'Laudo':'exm_laudo_texto',
                        'laudo_tratado':'exm_laudo_texto_tratado',
                        'resultado':'exm_laudo_resultado',
                        'achados':'exm_laudo_achados'
                        })

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{OUTPUT_TABLENAME} (
  record_id STRING,
  id_pct STRING,
  idunidade STRING,
  unidade STRING,
  pct_nome STRING,
  pct_cpf STRING,
  pct_sexo STRING,
  pct_datanasc DATE,
  idade_paciente INT,
  telefonePacienteDDD STRING,
  telefonePaciente STRING,
  exm_data DATE,
  exm_mod STRING,
  exm_titulo STRING,
  exm_tipo STRING,
  exm_an STRING,
  exm_status STRING,
  exm_laudo_dataliber DATE,
  exm_laudo_texto STRING,
  exm_laudo_texto_tratado STRING,
  exm_laudo_resultado STRING,
  exm_laudo_achados STRING,
  fl_relevante INT,
  dataExecucaoModelo DATE
)
USING delta
COMMENT 'Laudos processados e enriquecidos com NLP para análise de cólon';
""")

In [0]:
df_laudos_spark = spark.createDataFrame(df_laudos)

In [0]:
df_laudos_spark.createOrReplaceTempView("vw_colon_tb_02")

In [0]:
spark.sql(f"""
INSERT INTO {CATALOG}.{OUTPUT_TABLENAME}
SELECT s.*
FROM vw_colon_tb s
LEFT ANTI JOIN {CATALOG}.{OUTPUT_TABLENAME} t
    ON  t.idunidade = s.idunidade
    AND t.id_pct = s.id_pct
    AND CAST(t.exm_data AS DATE) = s.exm_data
    AND t.exm_an = s.exm_an
""")

In [0]:
display(df_laudos)

In [0]:
%sql
describe vw_colon_tb

In [0]:
cols_destino = spark.table("hive_metastore.ia.dev_tb_diamond_mod_colon_saida_conferencia").columns
cols_view = spark.table("vw_colon_tb").columns

set(cols_destino) - set(cols_view)


In [0]:
%sql
DROP TABLE hive_metastore.ia.dev_tb_diamond_mod_colon_saida_conferencia;


In [0]:
%sql
select 
*
from
ia.dev_tb_diamond_mod_colon_saida_conferencia